In [ ]:
# Import libraries 

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor 
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler


In [ ]:
# Load Dataset & Preprocessing

df = pd.read_csv("house_prices_cleaned.csv")

numeric_features = ["LotArea", "OverallQual", "OverallCond", "GrLivArea", "GarageCars"] 
categ_features = ["Neighborhood", "Foundation"]

X = df[numeric_features + categ_features].copy()
y = df["SalePrice"]

X[["LotArea", "GrLivArea"]] = X[["LotArea", "GrLivArea"]].astype(float)

scaler = MinMaxScaler()
X[["LotArea", "GrLivArea"]] = scaler.fit_transform(X[["LotArea", "GrLivArea"]])

X = pd.get_dummies(X, columns=categ_features, drop_first=True)


In [ ]:
# Train Models 

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Random Forest Regressor 

rf_model = RandomForestRegressor(random_state=42, n_estimators=100)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print("Random Forest:")
print(f"MAE: {mae_rf:.2f}")
print(f"RMSE: {rmse_rf:.2f}")
print(f"R²: {r2_rf:.2f}\n") 


In [ ]:
# Define Hyperparameter Grid & Run GridSearchCV

# Define the grid of parameters 

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'max_features': ['log2', 'sqrt'], 
    'min_samples_split': [2, 5, 10] 
}

# GridSearchCV Setup 

grid_search = GridSearchCV(
    estimator = RandomForestRegressor(random_state = 42),
    param_grid = param_grid, 
    cv = 5,
    scoring = 'r2', 
    n_jobs = -1
)

# Run grid search 
grid_search.fit(X_train, y_train) 

# Best hyperparameters
print("Best Parameters Found:", grid_search.best_params_)

# Best Models 
best_rf = grid_search.best_estimator_

# Convert GridSearchCV results to a DataFrame
results = pd.DataFrame(grid_search.cv_results_)

# Show hyperparameters and their mean R² score
results[['params', 'mean_test_score']].sort_values(by='mean_test_score', ascending=False).head(10)


In [ ]:
# Evaluate After Tuning 

# Predictions with tuned model 
y_pred_best = best_rf.predict(X_test)

# Metrics 
mae_best = mean_absolute_error(y_test, y_pred_best)
rmse_best = np.sqrt(mean_squared_error(y_test, y_pred_best))
r2_best = r2_score(y_test, y_pred_best)

print("\nRandom Forest After Tuning:")
print(f"MAE: {mae_best:.2f}")
print(f"RMSE: {rmse_best:.2f}")
print(f"R²: {r2_best:.2f}")


In [ ]:
# Visualization (Before vs After)

# Compare Metrics 

metrics = ["MAE", "RMSE", "R²"]
before = [mae_rf, rmse_rf, r2_rf]
after = [mae_best, rmse_best, r2_best]

x = np.arange(len(metrics)) 
width = 0.35

plt.figure(figsize=(8,6))
plt.bar(x - width/2, before, width, label="Before Tuning")
plt.bar(x + width/2, after, width, label="After Tuning")

plt.xticks(x, metrics)
plt.ylabel("Score")
plt.title("Random Forest Performance: Before vs After Hyperparameter Tuning")
plt.legend()
plt.show()


## Reflection

- The hyperparameters that made the biggest difference were 'n_estimators' and 'max_depth'
- Tuning improved performance because GridSearchCV systematically tested combinations of hyperparameters
- Trade-off: it took more time to run (since it trained many models), but the accuracy was slightly better
